# Chapter 4 · Quantum Mechanics for Materials — live notebook

This notebook runs entirely in your browser via the Pyodide kernel.
All state is saved to your browser's local storage; nothing is sent to a server.

Back to the chapter: <https://dongzhaohe321418-lab.github.io/materials-simulation-handbook/ch04-quantum/>

Finite-difference solution of two textbook problems: the particle in a box (analytical comparison) and the quantum harmonic oscillator (spotting the equal level spacing).

Only Pyodide-compatible packages are used (numpy, scipy, matplotlib, ipywidgets).


## 4.3 Particle in a 1D box — finite-difference diagonalisation

Build the Hamiltonian as a tridiagonal matrix and diagonalise. Compare against the closed form $E_n = n^2 \pi^2 \hbar^2 / (2 m L^2)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

HBAR = 1.054_571_817e-34
M_E = 9.109_383_7e-31
EV = 1.602_176_634e-19

def build_hamiltonian(n_grid, box_length, potential=None, mass=M_E):
    h = box_length / (n_grid + 1)
    x = np.linspace(h, box_length - h, n_grid)
    pref = HBAR ** 2 / (2.0 * mass * h ** 2)
    main = 2.0 * pref * np.ones(n_grid)
    off = -pref * np.ones(n_grid - 1)
    H = np.diag(main) + np.diag(off, k=1) + np.diag(off, k=-1)
    if potential is not None:
        H = H + np.diag(potential)
    return x, H

L = 1.0e-9
N = 400
x, H = build_hamiltonian(N, L)
eigvals, eigvecs = np.linalg.eigh(H)
h = L / (N + 1)
eigvecs = eigvecs / np.sqrt(h)

print(f'{"n":>3} {"E_num (eV)":>12} {"E_ana (eV)":>12} {"rel err":>10}')
for n in range(1, 5):
    e_ana = (n ** 2 * np.pi ** 2 * HBAR ** 2) / (2.0 * M_E * L ** 2)
    e_num = eigvals[n - 1]
    rel = abs(e_num - e_ana) / e_ana
    print(f'{n:>3d} {e_num / EV:>12.6f} {e_ana / EV:>12.6f} {rel:>10.2e}')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 5), sharex=True)
for n, ax in zip(range(1, 5), axes.ravel()):
    psi_ana = np.sqrt(2.0 / L) * np.sin(n * np.pi * x / L)
    psi_num = eigvecs[:, n - 1]
    if np.dot(psi_num, psi_ana) < 0:
        psi_num = -psi_num
    ax.plot(x * 1e9, psi_ana, 'k-', lw=2, label='analytic')
    ax.plot(x * 1e9, psi_num, 'r--', lw=1.2, label='finite-diff')
    ax.set_title(f'n = {n}, E = {eigvals[n - 1] / EV:.3f} eV')
    ax.set_xlabel('x (nm)')
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()


## 4.4 Harmonic oscillator — same code, different potential

Replace V(x) with $\tfrac{1}{2} m \omega^2 x^2$; expect equally spaced levels $E_n = \hbar \omega (n + 1/2)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Atomic-ish units: hbar = m = omega = 1
N = 600
L = 20.0       # box from -L/2 to L/2
h = L / (N + 1)
x = np.linspace(-L / 2 + h, L / 2 - h, N)
V = 0.5 * x ** 2
pref = 1.0 / (2.0 * h ** 2)
main = 2.0 * pref + V
off = -pref * np.ones(N - 1)
H = np.diag(main) + np.diag(off, k=1) + np.diag(off, k=-1)
vals, vecs = np.linalg.eigh(H)
print('first six levels:', np.round(vals[:6], 4))
print('expected n+1/2  :', np.arange(6) + 0.5)

fig, ax = plt.subplots(figsize=(6, 4))
for n in range(4):
    psi = vecs[:, n] / np.sqrt(h)
    ax.plot(x, psi + vals[n], label=f'n={n}')
ax.plot(x, 0.5 * x ** 2, 'k', lw=0.7)
ax.set_ylim(-0.2, 4.5)
ax.set_xlim(-5, 5)
ax.set_xlabel('x')
ax.set_ylabel('E (and shifted psi)')
ax.legend(fontsize=8)
ax.set_title('Quantum harmonic oscillator')
plt.show()
